# TimeGo ML Overload/Plateau Prototype

Backtests the shrinkage-regression overload model and slope-based plateau classifier (`ml-prototype/src/`) against the current rule-based baseline, using real exported `SetLog` data and the OpenPowerlifting bulk dataset. See `docs/superpowers/specs/2026-08-11-timego-ml-overload-plateau-prototype-design.md` for the full design.

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
from src.epley import estimate_one_rep_max
from src.priors import build_population_prior, LIFT_CATEGORY_COLUMNS
from src.backtest import backtest_exercise, summarize_backtest

opl_df = pd.read_csv("data/openpowerlifting.csv")
setlog_df = pd.read_csv("data/setlog_export.csv")
setlog_df["one_rep_max"] = setlog_df.apply(
    lambda row: estimate_one_rep_max(row["weightKg"], row["reps"]), axis=1
)
setlog_df = setlog_df.sort_values(["exerciseName", "timestamp"])

print(f"OpenPowerlifting rows (Raw equipment): {len(opl_df)}")
print(f"Personal SetLog rows: {len(setlog_df)}")
print(f"Distinct exercises logged: {setlog_df['exerciseName'].nunique()}")
setlog_df.groupby("exerciseName").size().sort_values(ascending=False)


OpenPowerlifting rows (Raw equipment): 1896248
Personal SetLog rows: 19
Distinct exercises logged: 9


exerciseName
Cable Crossover              3
Tricep Pushdown              3
Dip                          3
Flutter Kick                 2
Dumbbell Bench Press         2
Overhead Tricep Extension    2
Incline Dumbbell Press       2
Archer Push-Up               1
Push-Up                      1
dtype: int64

## Exercise-to-lift-category mapping

The OpenPowerlifting prior only applies to exercises mappable to a barbell squat/bench/deadlift (spec Section 2/Out of scope). As of this export, none of the logged exercises are barbell compounds — they're dumbbell/cable/bodyweight work — so this map is intentionally empty for now. Add entries here once a barbell squat, bench, or deadlift variant gets logged, e.g. `"Barbell Back Squat": "squat"`.

In [2]:
YOUR_SEX = None  # set to "M" or "F" once a mapped compound lift exists and a prior is actually needed
YOUR_BODYWEIGHT_KG = 61.8  # latest logged BodyMetric weight

LIFT_CATEGORY_BY_EXERCISE = {
    # e.g. "Barbell Back Squat": "squat", "Conventional Deadlift": "deadlift"
}


In [3]:
all_results = {}
skipped_too_short = []

for exercise_name, group in setlog_df.groupby("exerciseName"):
    one_rms = group["one_rep_max"].tolist()
    lift_category = LIFT_CATEGORY_BY_EXERCISE.get(exercise_name)
    prior = None
    if lift_category is not None and YOUR_SEX is not None:
        prior = build_population_prior(opl_df, lift_category, YOUR_SEX, YOUR_BODYWEIGHT_KG)

    results = backtest_exercise(one_rms, population_prior=prior)
    if results:
        all_results[exercise_name] = summarize_backtest(results)
    else:
        skipped_too_short.append((exercise_name, len(one_rms)))

if skipped_too_short:
    print("Skipped (not enough history yet, need > 3 sets to backtest):")
    for name, n in skipped_too_short:
        print(f"  {name}: {n} set(s) logged")

if all_results:
    report = pd.DataFrame(all_results).T
    report["ml_wins"] = report["ml_mae"] < report["baseline_mae"]
else:
    report = pd.DataFrame(columns=["ml_mae", "baseline_mae", "n", "ml_wins"])
    print("No exercise has more than 3 logged sets yet -- nothing to backtest. "
          "Re-run this notebook after logging more sets.")

report


Skipped (not enough history yet, need > 3 sets to backtest):
  Archer Push-Up: 1 set(s) logged
  Cable Crossover: 3 set(s) logged
  Dip: 3 set(s) logged
  Dumbbell Bench Press: 2 set(s) logged
  Flutter Kick: 2 set(s) logged
  Incline Dumbbell Press: 2 set(s) logged
  Overhead Tricep Extension: 2 set(s) logged
  Push-Up: 1 set(s) logged
  Tricep Pushdown: 3 set(s) logged
No exercise has more than 3 logged sets yet -- nothing to backtest. Re-run this notebook after logging more sets.


,ml_mae,baseline_mae,n,ml_wins


In [4]:
if report.empty:
    print("Verdict: not enough data to compare yet. "
          f"Current max history on any single exercise: {setlog_df.groupby('exerciseName').size().max()} sets "
          "(need at least 4 to backtest one held-out point). Log more sets and re-run.")
else:
    overall_ml_mae = report["ml_mae"].mean()
    overall_baseline_mae = report["baseline_mae"].mean()
    win_rate = report["ml_wins"].mean()

    print(f"ML model overall MAE: {overall_ml_mae:.2f}")
    print(f"Baseline overall MAE: {overall_baseline_mae:.2f}")
    print(f"ML model beats baseline on {win_rate:.0%} of exercises")
    print()
    print("Verdict: proceed to Kotlin port" if overall_ml_mae < overall_baseline_mae
          else "Verdict: baseline still wins, do not port yet")


Verdict: not enough data to compare yet. Current max history on any single exercise: 3 sets (need at least 4 to backtest one held-out point). Log more sets and re-run.
